# Corrected full baseline reproduction
This notebook is disabled by default. Select one corrected full configuration, pin model/tokenizer revisions if the result will be reported, enable a Kaggle GPU, and set `RUN_HEAVY=True`. Checkpoint selection uses validation macro F1; the test split is evaluated only after selection. Gold multi-task configurations intentionally refuse execution pending supervisor confirmation.

In [ ]:
from pathlib import Path
import json, subprocess, sys

RUN_HEAVY = False
CONFIG_NAME = 'corrected_full_mdistilbert_single_task.json'
# Other choices: corrected_full_xlmr_single_task.json.
# The *_multitask_gold.json configurations remain blocked by design.

def find_repo():
    candidates = [Path('/kaggle/working/Capstone-Project')]
    candidates.extend(Path('/kaggle/input').glob('*/Capstone-Project'))
    candidates.extend(path for path in Path('/kaggle/input').glob('*') if (path / 'corrected_pipeline').is_dir())
    for candidate in candidates:
        if (candidate / 'corrected_pipeline' / 'runner.py').is_file():
            return candidate
    raise FileNotFoundError('Attach the complete Capstone-Project repository to this Kaggle notebook.')

if not RUN_HEAVY:
    print('Guard active: full reproduction is disabled; no model, checkpoint, or dataset was loaded.')
else:
    import torch
    if not torch.cuda.is_available():
        raise RuntimeError('Enable a Kaggle GPU before full reproduction.')
    repo = find_repo()
    config_path = repo / 'configs' / CONFIG_NAME
    config = json.loads(config_path.read_text(encoding='utf-8'))
    assert config['run_kind'] == 'full'
    if config['execution']['blocked']:
        raise RuntimeError('Configuration is blocked: ' + config['execution']['blocked_reason'])
    missing = [path for path in config['dataset']['paths'].values() if not Path(path).is_file()]
    if missing:
        raise FileNotFoundError('Missing canonical Kaggle files: ' + ', '.join(missing))
    subprocess.run([sys.executable, '-m', 'compileall', '-q', 'corrected_pipeline', 'tests_stage1a'], cwd=repo, check=True)
    subprocess.run([sys.executable, '-m', 'unittest', 'discover', '-s', 'tests_stage1a', '-v'], cwd=repo, check=True)
    subprocess.run([sys.executable, '-m', 'corrected_pipeline.runner', '--config', str(config_path)], cwd=repo, check=True)